# 04. Avaluació del Model ARIMAX (Filtre de Kalman i Farmacocinètica)

En aquest quadern implemente l'algoritme ARIMAX, incorporant variables exògenes (ingesta de carbohidrats i injeccions d'insulina) a l'estructura base del model ARIMA(1,1,0). Configure l'entorn de treball i establisc els paràmetres globals de l'experiment, mantenint la finestra de 11 hores de context i 1 hora de predicció.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings, time
warnings.filterwarnings('ignore')

# ============================================================================
# PARÀMETRES CONFIGURABLES
# ============================================================================
MOSTRES_WARMUP = 132   # 11h × 12 mostres/h
MOSTRES_PRED   = 12    # 1h × 12 mostres/h
SALT_FINESTRA  = 1     # 5 min per pas (ajustable)

K_LAGS       = 6    # retards de bolus/carbs: 6 passos = 30 min
IOB_HALFLIFE = 18   # semivida IOB: 18 mostres = 90 min (insulina ràpida)

COLS_EXOG = (
    ['carbs', 'bolus', 'iob_exp']
    + [f'bolus_lag_{i}' for i in range(1, K_LAGS + 1)]
    + [f'carbs_lag_{i}' for i in range(1, K_LAGS + 1)]
)  # 15 variables exògenes totals

### 1. Enginyeria de Característiques (Feature Engineering)

Cree les variables exògenes necessàries per a dotar el model de context fisiològic. Genere columnes de retards (*lags*) per a capturar l'efecte diferit de la insulina i la ingesta nutricional. A més, calcule l'Insulina Activa (IOB) mitjançant un filtre de decaïment exponencial per a simular la farmacocinètica de la insulina d'acció ràpida. Aplique aquestes transformacions respectant de manera estricta els límits de cada segment continu per a evitar contaminació temporal.

In [16]:
def enginyeria_caracteristiques(df, k_lags=K_LAGS, iob_halflife=IOB_HALFLIFE):
    df = df.copy().sort_values(['pacient_id', 'segment_num', 'timestamp'])

    # 1. Lags de bolus i carbs (últims K passos = últims K×5 min)
    for lag in range(1, k_lags + 1):
        df[f'bolus_lag_{lag}'] = (
            df.groupby(['pacient_id', 'segment_num'])['bolus']
            .shift(lag).fillna(0)
        )
        df[f'carbs_lag_{lag}'] = (
            df.groupby(['pacient_id', 'segment_num'])['carbs']
            .shift(lag).fillna(0)
        )

    # 2. IOB (Insulin on Board) amb decaïment exponencial
    alpha = 1 - np.exp(-np.log(2) / iob_halflife)

    def iob_exp_decay(series):
        result, vals = np.zeros(len(series)), series.values
        for i in range(len(vals)):
            result[i] = vals[i] if i == 0 else vals[i] + (1 - alpha) * result[i - 1]
        return pd.Series(result, index=series.index)

    df['iob_exp'] = (
        df.groupby(['pacient_id', 'segment_num'])['bolus']
        .transform(iob_exp_decay)
    )
    return df

# Càrrega de dades i aplicació
df_train = pd.read_csv("dades_preprocessades/dataset_entrenament_net.csv")
df_test  = pd.read_csv("dades_preprocessades/dataset_test_net.csv")

df_train = enginyeria_caracteristiques(df_train)
df_test  = enginyeria_caracteristiques(df_test)

min_mostres_train = df_train.groupby(['pacient_id', 'segment_num']).size().min()
min_mostres_test  = df_test.groupby(['pacient_id', 'segment_num']).size().min()

assert min_mostres_train >= 144, f"Error: Hi ha segments curts en Train ({min_mostres_train} mostres)."
assert min_mostres_test  >= 144, f"Error: Hi ha segments curts en Test ({min_mostres_test} mostres)."
print("✅ Validació correcta: Tots els segments compleixen el llindar mínim de 12h.")
print(f"Feature engineering completat. Variables exògenes ({len(COLS_EXOG)}): {COLS_EXOG}")

✅ Validació correcta: Tots els segments compleixen el llindar mínim de 12h.
Feature engineering completat. Variables exògenes (15): ['carbs', 'bolus', 'iob_exp', 'bolus_lag_1', 'bolus_lag_2', 'bolus_lag_3', 'bolus_lag_4', 'bolus_lag_5', 'bolus_lag_6', 'carbs_lag_1', 'carbs_lag_2', 'carbs_lag_3', 'carbs_lag_4', 'carbs_lag_5', 'carbs_lag_6']


### 2. Calibració de Coeficients ARIMAX

Ajuste els paràmetres del model ARIMAX(1,1,0) per a cada pacient. Per a mantenir la coherència metodològica amb l'avaluació de la línia base, realitze l'entrenament exclusivament sobre el segment continu més llarg disponible en el conjunt d'entrenament de cada subjecte. Extraig i emmagatzeme els coeficients resultants, incloent-hi l'efecte de l'IOB i dels carbohidrats.

In [17]:
print("\n1. Calibrant coeficients ARIMAX(1,1,0) per a cada pacient...")

dic_coefs_pacient = {}
resultats_coefs   = []

mides_segments      = df_train.groupby(['pacient_id', 'segment_num']).size().reset_index(name='longitud')
idx_max             = mides_segments.groupby('pacient_id')['longitud'].idxmax()
segments_mes_llargs = mides_segments.loc[idx_max]

for _, row in segments_mes_llargs.iterrows():
    pacient = row['pacient_id']
    seg_num = row['segment_num']

    df_seg = df_train[
        (df_train['pacient_id'] == pacient) &
        (df_train['segment_num'] == seg_num)
    ].sort_values('timestamp')

    y    = df_seg['glucose_imputed'].values
    exog = df_seg[COLS_EXOG].fillna(0) 

    model = ARIMA(y, exog=exog, order=(1, 1, 0))
    res   = model.fit()
    dic_coefs_pacient[pacient] = res.params

    pnames = list(res.param_names)
    resultats_coefs.append({
        'Pacient'     : pacient,
        'AR(1)'       : round(res.params[pnames.index('ar.L1')], 4),
        'Sigma2'      : round(res.params[pnames.index('sigma2')], 4),
        'Coef_bolus'  : round(res.params[pnames.index('bolus')], 4),
        'Coef_carbs'  : round(res.params[pnames.index('carbs')], 4),
        'Coef_IOB'    : round(res.params[pnames.index('iob_exp')], 4),
    })
    
df_coefs_final = pd.DataFrame(resultats_coefs)
print("Calibració finalitzada i coeficients emmagatzemats.")


1. Calibrant coeficients ARIMAX(1,1,0) per a cada pacient...
Calibració finalitzada i coeficients emmagatzemats.


### 3. Avaluació en Test (Finestra Lliscant i Oracle Mode)

Avalue la capacitat predictiva del model mitjançant una validació de pas endavant. Implemente una finestra lliscant que utilitza 11 hores de context per a actualitzar els estats del filtre de Kalman. Per a establir el límit teòric de rendiment de l'algoritme (*upper bound*), proporcione les variables exògenes futures reals (Oracle Mode) a la funció de predicció durant el pronòstic a 60 minuts.

In [18]:
print("\n2. Avaluant ARIMAX(1,1,0) en Test amb finestra lliscant...")
start_time       = time.time()
prediccions_test = []

for pacient in df_test['pacient_id'].unique():
    df_pacient     = df_test[df_test['pacient_id'] == pacient]
    params_pacient = dic_coefs_pacient[pacient]

    for seg_num, df_seg in df_pacient.groupby('segment_num'):
        df_seg    = df_seg.sort_values('timestamp').reset_index(drop=True)
        y_test    = df_seg['glucose_imputed'].values
        exog_seg  = df_seg[COLS_EXOG].fillna(0).values  
        n_mostres = len(y_test)

        for i in range(0, n_mostres - MOSTRES_WARMUP - MOSTRES_PRED + 1, SALT_FINESTRA):
            y_warmup     = y_test[i : i + MOSTRES_WARMUP]
            exog_warmup  = exog_seg[i : i + MOSTRES_WARMUP]
            exog_futur   = exog_seg[i + MOSTRES_WARMUP : i + MOSTRES_WARMUP + MOSTRES_PRED]
            y_real_futur = y_test[i + MOSTRES_WARMUP : i + MOSTRES_WARMUP + MOSTRES_PRED]

            # Actualitze estats del model ARIMAX
            model_eval = ARIMA(y_warmup, exog=exog_warmup, order=(1, 1, 0))
            res_eval   = model_eval.filter(params_pacient)

            # Predicció passant les exògenes futures reals
            preds     = res_eval.forecast(steps=MOSTRES_PRED, exog=exog_futur)
            pred_vals = preds.values if hasattr(preds, 'values') else np.array(preds)

            # Timestamp de l'última observació del context (moment en què es fa la predicció)
            timestamp_pred = df_seg['timestamp'].iloc[i + MOSTRES_WARMUP - 1]

            prediccions_test.append({
                'Pacient'  : pacient,
                'Timestamp_Pred': timestamp_pred,
                'Real_15m' : y_real_futur[2],  'Pred_15m': pred_vals[2],
                'Real_30m' : y_real_futur[5],  'Pred_30m': pred_vals[5],
                'Real_45m' : y_real_futur[8],  'Pred_45m': pred_vals[8],
                'Real_60m' : y_real_futur[11], 'Pred_60m': pred_vals[11],
            })

df_res     = pd.DataFrame(prediccions_test)
temps_exec = time.time() - start_time
print(f"   Completat en {temps_exec/60:.2f} min. Finestres: {len(df_res)}")


2. Avaluant ARIMAX(1,1,0) en Test amb finestra lliscant...
   Completat en 6.25 min. Finestres: 11394


### 4. Resum de Mètriques globals i Exportació

Es presenten els resultats definitius de l'avaluació. Es calculen les mètriques RMSE, MAE i MARD (aplicant protecció matemàtica) per als horitzons fixats, i s'exporten els valors obtinguts juntament amb la taula de coeficients individuals per a l'anàlisi clínica.

In [19]:
def calcular_metriques(real, pred):
    rmse = np.sqrt(mean_squared_error(real, pred))
    mae  = mean_absolute_error(real, pred)
    mard = np.nanmean(np.abs((real - pred) / np.where(real == 0, np.nan, real))) * 100
    return rmse, mae, mard

taula_metriques = []
for h in ['15m', '30m', '45m', '60m']:
    rmse, mae, mard = calcular_metriques(df_res[f'Real_{h}'], df_res[f'Pred_{h}'])
    taula_metriques.append({'Horitzó': h, 'RMSE (mg/dL)': round(rmse,2),
                            'MAE (mg/dL)': round(mae,2), 'MARD (%)': round(mard,2)})

df_metriques = pd.DataFrame(taula_metriques)

print("\n" + "="*60)
print(" TAULA 1: RESULTATS GLOBALS EN TEST (ARIMAX 1,1,0)")
print("="*60)
print(df_metriques.to_string(index=False))

print("\n" + "="*60)
print(" TAULA 2: COEFICIENTS INDIVIDUALITZATS PER PACIENT")
print("="*60)
print(df_coefs_final.to_string(index=False))

# Exportació
df_res.to_csv("resultats_metrics/prediccions_arimax_test.csv", index=False)
df_metriques.to_csv("resultats_metrics/arimax_metriques_test.csv", index=False)
df_coefs_final.to_csv("resultats_metrics/arimax_coeficients_pacients.csv", index=False)
print("\nExportació completada.")


 TAULA 1: RESULTATS GLOBALS EN TEST (ARIMAX 1,1,0)
Horitzó  RMSE (mg/dL)  MAE (mg/dL)  MARD (%)
    15m         12.62         7.86      5.06
    30m         21.18        14.33      9.22
    45m         28.16        19.88     12.82
    60m         34.04        24.60     15.99

 TAULA 2: COEFICIENTS INDIVIDUALITZATS PER PACIENT
 Pacient  AR(1)  Sigma2  Coef_bolus  Coef_carbs  Coef_IOB
     559 0.5939 26.0307    -10.4590      0.0039   10.9777
     563 0.6012 12.8215     -1.0696     -0.0237    0.9858
     570 0.6585 10.7874     -0.6390      0.0130    0.8377
     575 0.7034  8.6943    -12.1970     -0.0275   12.8094
     588 0.6642 11.9660     -2.5266     -0.0053    2.3740
     591 0.7663  9.4941     -0.9378     -0.0141    1.0565

Exportació completada.


### 5. Avaluació Clínica: Detecció d'Hipoglucèmies

Per a completar l'avaluació de l'ARIMAX, traduïsc les prediccions contínues a un format de classificació binària establint el llindar clínic de risc en <= 70 mg/dL. Amb aquesta anàlisi, calcule el rendiment del model a l'hora de detectar i anticipar hipoglucèmies (Sensibilitat, Precisió i F1-Score) basant-me en la matriu de confusió de cada finestra temporal. L'objectiu és evidenciar com la incorporació de les variables exògenes i la informació farmacocinètica (IOB) ajuda a reduir els falsos negatius crítics a llarg termini. Finalment, exporte la taula resultant per a la seua inclusió a la memòria.

In [20]:
LLINDAR_HIPO = 70.0
resultats_hipo = []

for h in ['15m', '30m', '45m', '60m']:
    # Extracció de matrius per a l'horitzó corresponent
    y_true = df_res[f'Real_{h}'].values
    y_pred = df_res[f'Pred_{h}'].values

    # Binarització de les prediccions segons el llindar clínic
    real_hipo = (y_true <= LLINDAR_HIPO)
    pred_hipo = (y_pred <= LLINDAR_HIPO)

    # Càlcul de la Matriu de Confusió
    tp = np.sum(real_hipo & pred_hipo)        # Veritables Positius
    fn = np.sum(real_hipo & ~pred_hipo)       # Falsos Negatius
    fp = np.sum(~real_hipo & pred_hipo)       # Falsos Positius
    tn = np.sum(~real_hipo & ~pred_hipo)      # Veritables Negatius

    # Càlcul de mètriques amb prevenció de divisió per zero
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Acumulació de resultats
    resultats_hipo.append({
        'Horitzó'                    : h,
        'Casos_Hipo_Reals'           : int(np.sum(real_hipo)),
        'TP (Detectats)'             : int(tp),
        'FN (Omesos)'                : int(fn),
        'FP (Falses Alarmes)'        : int(fp),
        'Sensibilitat / Recall (%)'  : round(recall * 100, 2),
        'Precisió (%)'               : round(precision * 100, 2),
        'F1-Score (%)'               : round(f1 * 100, 2)
    })

df_hipo = pd.DataFrame(resultats_hipo)

print("\n" + "="*80)
print(" TAULA 3: DETECCIÓ D'HIPOGLUCÈMIES — MODEL ARIMAX")
print("="*80)
print(df_hipo.to_string(index=False))

# Exportació dels resultats de classificació específics del model ARIMAX
df_hipo.to_csv("resultats_metrics/arimax_hipoglucemies.csv", index=False)
print("\nExportació de mètriques clíniques completada.")


 TAULA 3: DETECCIÓ D'HIPOGLUCÈMIES — MODEL ARIMAX
Horitzó  Casos_Hipo_Reals  TP (Detectats)  FN (Omesos)  FP (Falses Alarmes)  Sensibilitat / Recall (%)  Precisió (%)  F1-Score (%)
    15m               236             189           47                   76                      80.08         71.32         75.45
    30m               238             150           88                  143                      63.03         51.19         56.50
    45m               243             124          119                  195                      51.03         38.87         44.13
    60m               246              95          151                  229                      38.62         29.32         33.33

Exportació de mètriques clíniques completada.
